# 01 - Compréhension des données

Objectif : comprendre la structure du fichier source avant la préparation des données.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 50)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / 'data' / 'raw' / 'incident_event_log.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DATABASE_DIR = PROJECT_ROOT / 'data' / 'database'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DATABASE_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
df = pd.read_csv(RAW_DATA, na_values=['?'], low_memory=False)

print(f'Lignes du journal : {df.shape[0]:,}')
print(f'Colonnes : {df.shape[1]}')
print(f'Incidents uniques : {df['number'].nunique():,}')
df.head()

Lignes du journal : 141,712
Colonnes : 36
Incidents uniques : 24,918


,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,sys_created_by,sys_created_at,sys_updated_by,sys_updated_at,contact_type,location,category,subcategory,u_symptom,cmdb_ci,impact,urgency,priority,assignment_group,assigned_to,knowledge,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at
0,INC0000045,New,True,0,0,0,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 21,29/2/2016 01:23,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,NaN,2 - Medium,2 - Medium,3 - Moderate,Group 56,NaN,True,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
1,INC0000045,Resolved,True,0,0,2,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 642,29/2/2016 08:53,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,NaN,2 - Medium,2 - Medium,3 - Moderate,Group 56,NaN,True,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
2,INC0000045,Resolved,True,0,0,3,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 804,29/2/2016 11:29,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,NaN,2 - Medium,2 - Medium,3 - Moderate,Group 56,NaN,True,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
3,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 908,5/3/2016 12:00,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,NaN,2 - Medium,2 - Medium,3 - Moderate,Group 56,NaN,True,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
4,INC0000047,New,True,0,0,0,True,Caller 2403,Opened by 397,29/2/2016 04:40,Created by 171,29/2/2016 04:57,Updated by 746,29/2/2016 04:57,Phone,Location 165,Category 40,Subcategory 215,Symptom 471,NaN,2 - Medium,2 - Medium,3 - Moderate,Group 70,Resolver 89,True,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 81,1/3/2016 09:52,6/3/2016 10:00


## Structure du fichier

Le fichier source est un journal d'événements : un même incident peut apparaître sur plusieurs lignes. Cette information est importante, car les KPI doivent être calculés au niveau incident et non au niveau événement.

In [3]:
structure = pd.DataFrame({
    'indicateur': [
        'Lignes du journal',
        'Incidents uniques',
        'Lignes moyennes par incident',
        'Incidents avec plusieurs lignes'
    ],
    'valeur': [
        len(df),
        df['number'].nunique(),
        round(len(df) / df['number'].nunique(), 2),
        (df['number'].value_counts() > 1).sum()
    ]
})
structure

,indicateur,valeur
0,Lignes du journal,141712.00
1,Incidents uniques,24918.00
2,Lignes moyennes par incident,5.69
3,Incidents avec plusieurs lignes,24918.00


## Colonnes et valeurs manquantes

On vérifie les types et les valeurs manquantes pour repérer les colonnes à nettoyer avant l'analyse.

In [4]:
types_df = pd.DataFrame({
    'colonne': df.columns,
    'type': [str(t) for t in df.dtypes],
    'valeurs_manquantes': df.isna().sum().values,
    'taux_manquant_pct': (df.isna().mean().values * 100).round(2)
}).sort_values('valeurs_manquantes', ascending=False)

types_df.head(15)

,colonne,type,valeurs_manquantes,taux_manquant_pct
31,caused_by,object,141689,99.98
30,vendor,object,141468,99.83
19,cmdb_ci,object,141267,99.69
29,rfc,object,140721,99.30
28,problem_id,object,139417,98.38
10,sys_created_by,object,53076,37.45
11,sys_created_at,object,53076,37.45
18,u_symptom,object,32964,23.26
24,assigned_to,object,27496,19.40
23,assignment_group,object,14213,10.03


## Variables importantes

Pour ce projet, on garde surtout les SLA, les dates, la priorité, la catégorie, le groupe support, les réassignations et les réouvertures.

In [5]:
colonnes_cles = [
    'number', 'made_sla', 'opened_at', 'resolved_at', 'closed_at',
    'priority', 'impact', 'urgency', 'category', 'assignment_group',
    'reassignment_count', 'reopen_count'
]

df[colonnes_cles].head()

,number,made_sla,opened_at,resolved_at,closed_at,priority,impact,urgency,category,assignment_group,reassignment_count,reopen_count
0,INC0000045,True,29/2/2016 01:16,29/2/2016 11:29,5/3/2016 12:00,3 - Moderate,2 - Medium,2 - Medium,Category 55,Group 56,0,0
1,INC0000045,True,29/2/2016 01:16,29/2/2016 11:29,5/3/2016 12:00,3 - Moderate,2 - Medium,2 - Medium,Category 55,Group 56,0,0
2,INC0000045,True,29/2/2016 01:16,29/2/2016 11:29,5/3/2016 12:00,3 - Moderate,2 - Medium,2 - Medium,Category 55,Group 56,0,0
3,INC0000045,True,29/2/2016 01:16,29/2/2016 11:29,5/3/2016 12:00,3 - Moderate,2 - Medium,2 - Medium,Category 55,Group 56,0,0
4,INC0000047,True,29/2/2016 04:40,1/3/2016 09:52,6/3/2016 10:00,3 - Moderate,2 - Medium,2 - Medium,Category 40,Group 70,0,0


## Période couverte

La période doit être contrôlée avant de commenter une tendance. Si certains mois ont très peu d'incidents, il faut rester prudent dans l'interprétation.

In [6]:
opened_at = pd.to_datetime(df['opened_at'], format='%d/%m/%Y %H:%M', errors='coerce')
periode = pd.DataFrame({
    'indicateur': ['Première ouverture', 'Dernière ouverture'],
    'valeur': [opened_at.min(), opened_at.max()]
})
periode

,indicateur,valeur
0,Première ouverture,2016-02-29 01:16:00
1,Dernière ouverture,2017-02-16 14:17:00


## Conclusion

Le fichier contient les informations nécessaires pour analyser les SLA, les délais de traitement et la charge des groupes support.

Point clé pour la suite : le fichier brut est un journal d'événements avec plusieurs lignes possibles par incident. Il faut donc créer une table avec une seule ligne par incident avant de calculer les KPI métier.